# 3. Machine Learning + Deep Learning


In [ ]:
# @title Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scikit-learn torch -q
print("Packages installed")


In [ ]:
# @title Load Tutorial
import os

REPO = "/content/MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.isdir(REPO):
    %cd {REPO}
    !git fetch origin -q
    !git checkout main -q
    !git pull --ff-only -q
    print("Repo updated")
else:
    %cd /content
    !git clone {REPO_URL} -q
    %cd {REPO}
    print("Repo cloned")


## 3a) Training Conventional ML Models (KNN, RF, SVM)

In [ ]:
from pathlib import Path
import csv

import numpy as np
from IPython.display import Image, display

# scikit-learn imports for data prep and ML models
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from tutorial_utils.sections import ml_conv_1phase as s03a


def generate_conv_ml_1phase_data(synth_samples_per_formula=12, random_seed=42):
    """Step 1: generate synthetic one-phase training data + experimental test data."""
    s03a.SYNTH_SAMPLES_PER_FORMULA = synth_samples_per_formula
    s03a.RANDOM_SEED = random_seed

    rng = np.random.default_rng(random_seed)
    two_theta_grid = np.linspace(s03a.MIN_ANGLE, s03a.MAX_ANGLE, s03a.NUM_POINTS)

    refs = s03a.load_reference_sticks(sorted(s03a.REFERENCE_DIR.glob("*.cif")))
    X, y = s03a.build_synthetic_dataset(refs, two_theta_grid, rng)

    exp_files = sorted(s03a.EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([s03a.preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test = np.asarray([f.stem for f in exp_files])

    return {"X": X, "y": y, "X_test": X_test, "y_test": y_test, "random_seed": random_seed}


def split_conv_ml_1phase_data(dataset, val_fraction=0.20):
    """Step 2: split synthetic data into train/validation sets."""
    X_train, X_val, y_train, y_val = train_test_split(
        dataset["X"],
        dataset["y"],
        test_size=val_fraction,
        random_state=dataset["random_seed"],
        stratify=dataset["y"],
    )
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": dataset["X_test"],
        "y_test": dataset["y_test"],
        "random_seed": dataset["random_seed"],
    }


def train_knn_1phase(split_data, neighbor_values=(1, 3, 5, 7, 11)):
    """Step 3a: tune + train k-NN."""
    return s03a.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda n_neighbors: make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance"),
        ),
        param_grid=[{"n_neighbors": k} for k in neighbor_values],
    )


def train_random_forest_1phase(
    split_data,
    n_estimators=300,
    max_depth_options=(None, 15, 30),
    min_samples_leaf_options=(1, 2, 4),
):
    """Step 3b: tune + train Random Forest."""
    return s03a.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda max_depth, min_samples_leaf: RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=split_data["random_seed"],
            n_jobs=-1,
        ),
        param_grid=[
            {"max_depth": d, "min_samples_leaf": l}
            for d in max_depth_options
            for l in min_samples_leaf_options
        ],
    )


def train_svm_1phase(split_data, c_values=(1.0, 5.0, 10.0), gamma_values=("scale", 0.01, 0.001)):
    """Step 3c: tune + train SVM."""
    return s03a.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda C, gamma: make_pipeline(
            StandardScaler(),
            SVC(kernel="rbf", C=C, gamma=gamma),
        ),
        param_grid=[{"C": c, "gamma": g} for c in c_values for g in gamma_values],
    )


def evaluate_1phase_model(best_result, split_data):
    """Step 4: evaluate one trained model on experimental test patterns."""
    y_pred_test = best_result["model"].predict(split_data["X_test"])
    return {
        "val_acc": float(best_result["val_acc"]),
        "test_acc": float(accuracy_score(split_data["y_test"], y_pred_test)),
        "params": best_result["params"],
    }


def save_conv_ml_1phase_results(model_results, output_dir="outputs/ml/conv"):
    """Step 5: save summary plot + compact metric table."""
    s03a.OUTPUT_DIR = Path(output_dir)
    s03a.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    s03a.plot_accuracy_summary(model_results)

    metrics_file = s03a.OUTPUT_DIR / "model_test_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "val_acc", "test_acc", "best_params"])
        writer.writeheader()
        for name, result in model_results.items():
            writer.writerow(
                {
                    "model": name,
                    "val_acc": result["val_acc"],
                    "test_acc": result["test_acc"],
                    "best_params": str(result["params"]),
                }
            )

    print(f"Saved metrics: {metrics_file}")


This section uses **scikit-learn** models (k-NN, Random Forest, SVM) on synthetic one-phase training data.


Workflow:
1. Generate synthetic data and aligned experimental test data.
2. Split synthetic data into train/validation sets.
3. Train each model type separately and compare performance.


## Run the Demo


In [ ]:
# @title Run Conventional ML (1-Phase)
# Step 1: data generation
dataset_1phase = generate_conv_ml_1phase_data(synth_samples_per_formula=12)

# Step 2: split
split_1phase = split_conv_ml_1phase_data(dataset_1phase)

# Step 3 + 4: train and evaluate each scikit-learn model type
results_1phase = {
    "k-NN": evaluate_1phase_model(train_knn_1phase(split_1phase), split_1phase),
    "Random Forest": evaluate_1phase_model(train_random_forest_1phase(split_1phase), split_1phase),
    "SVM": evaluate_1phase_model(train_svm_1phase(split_1phase), split_1phase),
}

# Step 5: save outputs
save_conv_ml_1phase_results(results_1phase)

# Try on your own:
# 1) Generate more or fewer synthetic samples.
# dataset_1phase = generate_conv_ml_1phase_data(synth_samples_per_formula=24)
# dataset_1phase = generate_conv_ml_1phase_data(synth_samples_per_formula=6)
#
# 2) Change SVM search space.
# results_1phase["SVM"] = evaluate_1phase_model(
#     train_svm_1phase(split_1phase, c_values=(0.5, 1.0, 5.0), gamma_values=("scale", 0.01)),
#     split_1phase,
# )


## Example Output


In [ ]:
display(Image("outputs/ml/conv/model_accuracy_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3b) Conventional ML: Multiphase


In [ ]:
from pathlib import Path
import csv

import numpy as np
from IPython.display import Image, display

# Explicit scikit-learn imports for conventional multi-label models.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from tutorial_utils.sections import ml_multiphase as s03b


def generate_conv_ml_multiphase_data(
    synth_samples_per_formula=10,
    single_phase_fraction=0.15,
    random_seed=42,
):
    """Step 1: generate synthetic multiphase data + experimental test data."""
    s03b.SYNTH_SAMPLES_PER_FORMULA = synth_samples_per_formula
    s03b.SINGLE_PHASE_FRACTION = single_phase_fraction
    s03b.RANDOM_SEED = random_seed

    rng = np.random.default_rng(random_seed)
    two_theta_grid = np.linspace(s03b.MIN_ANGLE, s03b.MAX_ANGLE, s03b.NUM_POINTS)

    refs_by_formula = s03b.load_reference_sticks(sorted(s03b.REFERENCE_DIR.glob("*.cif")))
    all_formulas = sorted(refs_by_formula.keys())

    # Multiphase construction: weighted linear combinations of simulated single-phase components.
    X, y_label_lists = s03b.build_synthetic_multiphase_dataset(refs_by_formula, two_theta_grid, rng)
    mlb = MultiLabelBinarizer(classes=all_formulas)
    y_bin = mlb.fit_transform(y_label_lists)

    exp_files = sorted(s03b.EXPERIMENT_DIR.glob("*.xy"))
    X_test = np.asarray([s03b.preprocess_experimental_pattern(f, two_theta_grid) for f in exp_files])
    y_test_labels = [s03b.labels_from_filename(f.stem) for f in exp_files]
    y_test = mlb.transform(y_test_labels)

    return {"X": X, "y": y_bin, "X_test": X_test, "y_test": y_test, "random_seed": random_seed}


def split_conv_ml_multiphase_data(dataset, val_fraction=0.20):
    """Step 2: split synthetic multiphase data into train/validation sets."""
    X_train, X_val, y_train, y_val = train_test_split(
        dataset["X"],
        dataset["y"],
        test_size=val_fraction,
        random_state=dataset["random_seed"],
    )
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": dataset["X_test"],
        "y_test": dataset["y_test"],
        "random_seed": dataset["random_seed"],
    }


def train_knn_multiphase(split_data, neighbor_values=(1, 3, 5, 7, 11)):
    """Step 3a: tune + train one-vs-rest k-NN."""
    return s03b.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda n_neighbors: OneVsRestClassifier(
            make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance"))
        ),
        param_grid=[{"n_neighbors": k} for k in neighbor_values],
    )


def train_random_forest_multiphase(
    split_data,
    n_estimators=250,
    max_depth_options=(None, 15, 30),
    min_samples_leaf_options=(1, 2, 4),
):
    """Step 3b: tune + train one-vs-rest Random Forest."""
    return s03b.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda max_depth, min_samples_leaf: OneVsRestClassifier(
            RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                random_state=split_data["random_seed"],
                n_jobs=-1,
            )
        ),
        param_grid=[
            {"max_depth": d, "min_samples_leaf": l}
            for d in max_depth_options
            for l in min_samples_leaf_options
        ],
    )


def train_svm_multiphase(split_data, c_values=(1.0, 5.0, 10.0), gamma_values=("scale", 0.01, 0.001)):
    """Step 3c: tune + train one-vs-rest SVM."""
    return s03b.tune_model(
        split_data["X_train"],
        split_data["y_train"],
        split_data["X_val"],
        split_data["y_val"],
        model_builder=lambda C, gamma: OneVsRestClassifier(
            make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma, probability=True))
        ),
        param_grid=[{"C": c, "gamma": g} for c in c_values for g in gamma_values],
    )


def evaluate_multiphase_model(best_result, split_data):
    """Step 4: evaluate one model with its validation-picked threshold."""
    threshold = best_result["threshold"]
    test_scores = s03b.get_label_scores(best_result["model"], split_data["X_test"])
    precision, recall, f1_micro, _ = s03b.threshold_metrics(split_data["y_test"], test_scores, threshold)

    return {
        "threshold": float(threshold),
        "params": best_result["params"],
        "val_f1_micro": float(best_result["val_f1_micro"]),
        "test_precision_micro": float(precision),
        "test_recall_micro": float(recall),
        "test_f1_micro": float(f1_micro),
    }


def save_conv_ml_multiphase_results(model_results, output_dir="outputs/ml/multiphase"):
    """Step 5: save summary plot + compact metric table."""
    s03b.OUTPUT_DIR = Path(output_dir)
    s03b.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    s03b.plot_test_metric_summary(model_results)

    metrics_file = s03b.OUTPUT_DIR / "model_test_metrics.csv"
    with open(metrics_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["model", "threshold", "val_f1_micro", "test_precision_micro", "test_recall_micro", "test_f1_micro", "best_params"],
        )
        writer.writeheader()
        for name, result in model_results.items():
            writer.writerow(
                {
                    "model": name,
                    "threshold": result["threshold"],
                    "val_f1_micro": result["val_f1_micro"],
                    "test_precision_micro": result["test_precision_micro"],
                    "test_recall_micro": result["test_recall_micro"],
                    "test_f1_micro": result["test_f1_micro"],
                    "best_params": str(result["params"]),
                }
            )

    print(f"Saved metrics: {metrics_file}")


This section also uses **scikit-learn**, with one-vs-rest wrappers for multi-label phase prediction.

Important idea: synthetic multiphase patterns are built as weighted linear combinations of simulated single-phase component profiles (plus artifacts/noise).


Workflow:
1. Build synthetic multiphase data and binarize labels.
2. Split into train/validation sets.
3. Train k-NN, Random Forest, and SVM separately.
4. Compare micro-averaged test metrics.


## Run the Demo


In [ ]:
# @title Run Conventional ML (Multiphase)
# Step 1: data generation
dataset_multi = generate_conv_ml_multiphase_data(
    synth_samples_per_formula=10,
    single_phase_fraction=0.15,
)

# Step 2: split
split_multi = split_conv_ml_multiphase_data(dataset_multi)

# Step 3 + 4: train and evaluate each scikit-learn model type
results_multi = {
    "k-NN": evaluate_multiphase_model(train_knn_multiphase(split_multi), split_multi),
    "Random Forest": evaluate_multiphase_model(train_random_forest_multiphase(split_multi), split_multi),
    "SVM": evaluate_multiphase_model(train_svm_multiphase(split_multi), split_multi),
}

# Step 5: save outputs
save_conv_ml_multiphase_results(results_multi)

# Try on your own:
# 1) Increase synthetic mixture count.
# dataset_multi = generate_conv_ml_multiphase_data(synth_samples_per_formula=20)
#
# 2) Change single-phase fraction and compare precision/recall tradeoff.
# dataset_multi = generate_conv_ml_multiphase_data(single_phase_fraction=0.30)
# dataset_multi = generate_conv_ml_multiphase_data(single_phase_fraction=0.05)


## Example Output


In [ ]:
display(Image("outputs/ml/multiphase/model_test-metric_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3c) Neural Networks: 1-Phase


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_nn_1phase


def create_nn_1phase_demo(
    synth_samples_per_formula=12,
    hidden_layer_sizes=(128, 64),
    nn_max_iter=80,
):
    """3c) Dense NN for single-phase ID with key architecture controls."""
    # Note: this dense NN section uses scikit-learn's MLPClassifier backend.
    return run_nn_1phase(
        synth_samples_per_formula=synth_samples_per_formula,
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        show_steps=True,
    )


Train a dense neural network on synthetic single-phase patterns.


For live demos, keep epochs lower for speed.


## Run the Demo


In [ ]:
# @title Run NN (1-Phase)
create_nn_1phase_demo()

# Try on your own:
# 1) Make the network wider/deeper.
# create_nn_1phase_demo(hidden_layer_sizes=(256, 128, 64))
#
# 2) Run a shorter vs longer training budget.
# create_nn_1phase_demo(nn_max_iter=40)
# create_nn_1phase_demo(nn_max_iter=160)
#
# 3) Compare model capacity at fixed training budget.
# create_nn_1phase_demo(hidden_layer_sizes=(64, 32), nn_max_iter=80)


## Example Output


In [ ]:
display(Image("outputs/dl/nn_1phase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_1phase/nn_accuracy_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3d) Neural Networks: Multiphase


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_nn_multiphase


def create_nn_multiphase_demo(
    synth_samples_per_formula=8,
    hidden_layer_sizes=(128, 64),
    nn_max_iter=80,
    prediction_threshold=0.50,
):
    """3d) Dense multi-label NN with key architecture + threshold controls."""
    # Note: this dense NN section uses scikit-learn's MLPClassifier backend.
    return run_nn_multiphase(
        synth_samples_per_formula=synth_samples_per_formula,
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        prediction_threshold=prediction_threshold,
        show_steps=True,
    )


Train a dense multi-label network on synthetic mixtures.


For live demos, keep epochs lower for speed.


## Run the Demo


In [ ]:
# @title Run NN (Multiphase)
create_nn_multiphase_demo()

# Try on your own:
# 1) Increase model capacity.
# create_nn_multiphase_demo(hidden_layer_sizes=(256, 128))
#
# 2) Sweep decision threshold to trade precision vs recall.
# create_nn_multiphase_demo(prediction_threshold=0.35)
# create_nn_multiphase_demo(prediction_threshold=0.65)
#
# 3) Change training budget.
# create_nn_multiphase_demo(nn_max_iter=140)


## Example Output


In [ ]:
display(Image("outputs/dl/nn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/nn_multiphase/nn_test-metric_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3e) CNNs: Multiphase


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_multiphase


def create_cnn_multiphase_demo(
    synth_samples_per_formula=8,
    conv_channels=(16, 32),
    kernel_sizes=(7, 5),
    hidden_layer_sizes=(128, 64),
    nn_max_iter=20,
):
    """3e) 1D CNN multiphase demo with key architecture controls."""
    # Conv stack extracts local peak-shape features; dense head maps features to labels.
    return run_cnn_multiphase(
        synth_samples_per_formula=synth_samples_per_formula,
        conv_channels=conv_channels,
        kernel_sizes=kernel_sizes,
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        show_steps=True,
    )


Train a 1D CNN to capture local peak-shape context in multiphase patterns.


## Optional Pretrained Check


In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_multiphase.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo


In [ ]:
# @title Run CNN (Multiphase)
create_cnn_multiphase_demo()

# Try on your own:
# 1) Change convolutional capacity.
# create_cnn_multiphase_demo(conv_channels=(24, 48), kernel_sizes=(9, 5))
#
# 2) Change classifier-head size.
# create_cnn_multiphase_demo(hidden_layer_sizes=(256, 128))
#
# 3) Increase training epochs and compare stability.
# create_cnn_multiphase_demo(nn_max_iter=40)


## Example Output


In [ ]:
display(Image("outputs/dl/cnn_multiphase/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_multiphase/nn_test-metric_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3f) Ablation: No Augmentation


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_no_augmentation


def create_cnn_no_augmentation_demo(
    synth_samples_per_formula=8,
    conv_channels=(16, 32),
    hidden_layer_sizes=(128, 64),
    nn_max_iter=20,
):
    """3f) CNN no-augmentation ablation with key architecture controls."""
    # Kernel sizes are fixed here to keep this ablation focused on data effects.
    return run_cnn_no_augmentation(
        synth_samples_per_formula=synth_samples_per_formula,
        conv_channels=conv_channels,
        kernel_sizes=(7, 5),
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        show_steps=True,
    )


CNN baseline with simplified synthetic data (minimal augmentation).


## Optional Pretrained Check


In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_no_augmentation.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo


In [ ]:
# @title Run CNN (No Augmentation)
create_cnn_no_augmentation_demo()

# Try on your own:
# 1) Increase conv stack size.
# create_cnn_no_augmentation_demo(conv_channels=(24, 48))
#
# 2) Compare short vs longer training.
# create_cnn_no_augmentation_demo(nn_max_iter=10)
# create_cnn_no_augmentation_demo(nn_max_iter=40)
#
# 3) Shrink or expand dense head.
# create_cnn_no_augmentation_demo(hidden_layer_sizes=(64, 32))
# create_cnn_no_augmentation_demo(hidden_layer_sizes=(256, 128))


## Example Output


In [ ]:
display(Image("outputs/dl/cnn_no_augmentation/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_no_augmentation/nn_test-metric_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3g) Ablation: Random Shifts


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_cnn_random_shifts


def create_cnn_random_shifts_demo(
    synth_samples_per_formula=8,
    conv_channels=(16, 32),
    hidden_layer_sizes=(128, 64),
    nn_max_iter=20,
):
    """3g) CNN random-shifts ablation with key architecture controls."""
    # Keep kernel sizes fixed to isolate the effect of shift-only augmentation.
    return run_cnn_random_shifts(
        synth_samples_per_formula=synth_samples_per_formula,
        conv_channels=conv_channels,
        kernel_sizes=(7, 5),
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        show_steps=True,
    )


CNN baseline with shift-focused augmentation only.


## Optional Pretrained Check


In [ ]:
PRETRAINED_PATH = "data/pretrained/cnn_random_shifts.pt"
print("Found pretrained weights:", os.path.exists(PRETRAINED_PATH))

## Run the Demo


In [ ]:
# @title Run CNN (Random Shifts)
create_cnn_random_shifts_demo()

# Try on your own:
# 1) Increase conv stack size.
# create_cnn_random_shifts_demo(conv_channels=(24, 48))
#
# 2) Compare short vs longer training.
# create_cnn_random_shifts_demo(nn_max_iter=10)
# create_cnn_random_shifts_demo(nn_max_iter=40)
#
# 3) Shrink or expand dense head.
# create_cnn_random_shifts_demo(hidden_layer_sizes=(64, 32))
# create_cnn_random_shifts_demo(hidden_layer_sizes=(256, 128))


## Example Output


In [ ]:
display(Image("outputs/dl/cnn_random_shifts/nn_loss_curve.png"))
display(Image("outputs/dl/cnn_random_shifts/nn_test-metric_summary.png"))

## Next Steps
Continue to the next section below in this notebook.

## 3h) Mixture of Experts


In [ ]:
from IPython.display import Image, display

from tutorial_utils.runners import run_mixture_of_experts


def create_moe_demo(
    synth_samples_per_formula=6,
    conv_channels=(6, 12),
    hidden_layer_sizes=(16, 8),
    nn_max_iter=20,
    min_epochs=5,
    patience=4,
):
    """3h) Mixture-of-experts with key architecture + early-stopping controls."""
    # Keep kernels fixed so users focus on expert size and stopping behavior.
    return run_mixture_of_experts(
        synth_samples_per_formula=synth_samples_per_formula,
        conv_channels=conv_channels,
        kernel_sizes=(5, 3),
        hidden_layer_sizes=hidden_layer_sizes,
        nn_max_iter=nn_max_iter,
        min_epochs=min_epochs,
        patience=patience,
        show_steps=True,
    )


Train one lightweight expert per phase label, then combine their outputs.\nThis design is useful when phase count grows, because each expert stays small and training scales more gently.\n

## Optional Scaling Benchmark
Time the **single CNN classifier** vs **MoE-CNN** as the number of reference phases increases.
This benchmark measures training time only (not accuracy), and is intended to show scaling trends.


In [ ]:
# @title Benchmark CNN vs MoE Training-Time Scaling
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

from tutorial_utils.sections import dl_cnn_multiphase as sec_cnn
from tutorial_utils.sections import dl_moe as sec_moe


def _formula_to_cif_map(reference_dir=Path("data/reference_structures")):
    formula_to_cifs = {}
    for cif in sorted(reference_dir.glob("*.cif")):
        formula = cif.stem.split("_", 1)[0]
        formula_to_cifs.setdefault(formula, []).append(cif)
    return formula_to_cifs


def _build_subset_dataset(sec_module, cif_paths, synth_samples_per_formula=6, random_seed=42):
    """Build a small synthetic multi-label dataset for a subset of phases."""
    rng = np.random.default_rng(random_seed)
    two_theta_grid = np.linspace(sec_module.MIN_ANGLE, sec_module.MAX_ANGLE, sec_module.NUM_POINTS)
    refs_by_formula = sec_module.load_reference_sticks(cif_paths)
    formulas = sorted(refs_by_formula.keys())

    X = []
    y_labels = []

    for anchor in formulas:
        others = [f for f in formulas if f != anchor]
        for _ in range(synth_samples_per_formula):
            # Keep the same spirit as the existing generators, but handle small phase counts (e.g., n=2).
            if len(formulas) >= 3:
                n_components = int(
                    rng.choice(
                        [1, 2, 3],
                        p=[
                            sec_module.SINGLE_PHASE_FRACTION,
                            (1.0 - sec_module.SINGLE_PHASE_FRACTION) * 0.65,
                            (1.0 - sec_module.SINGLE_PHASE_FRACTION) * 0.35,
                        ],
                    )
                )
            else:
                n_components = int(
                    rng.choice(
                        [1, 2],
                        p=[sec_module.SINGLE_PHASE_FRACTION, 1.0 - sec_module.SINGLE_PHASE_FRACTION],
                    )
                )

            chosen_others = list(rng.choice(others, size=n_components - 1, replace=False)) if n_components > 1 else []
            labels = sorted([anchor] + chosen_others)

            profile = sec_module.simulate_multiphase_profile(two_theta_grid, labels, refs_by_formula, rng)
            X.append(profile)
            y_labels.append(labels)

    X = np.asarray(X)
    mlb = MultiLabelBinarizer(classes=formulas)
    y = mlb.fit_transform(y_labels)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=random_seed,
    )
    return X_train, X_val, y_train, y_val


def benchmark_cnn_vs_moe_scaling(
    phase_counts=(2, 4, 6, 8, 10),
    synth_samples_per_formula=6,
    cnn_epochs=12,
    moe_epochs=12,
    random_seed=42,
):
    formula_to_cifs = _formula_to_cif_map()
    all_formulas = sorted(formula_to_cifs.keys())
    counts = [n for n in phase_counts if 2 <= n <= len(all_formulas)]

    cnn_times = []
    moe_times = []

    # Save and restore module-level defaults after benchmarking.
    old_cnn_max_iter = sec_cnn.NN_MAX_ITER
    old_moe_max_iter = sec_moe.NN_MAX_ITER
    old_moe_min_epochs = sec_moe.NN_EARLY_STOPPING_MIN_EPOCHS
    old_moe_patience = sec_moe.NN_EARLY_STOPPING_PATIENCE

    sec_cnn.NN_MAX_ITER = cnn_epochs
    sec_moe.NN_MAX_ITER = moe_epochs
    sec_moe.NN_EARLY_STOPPING_MIN_EPOCHS = min(4, moe_epochs)
    sec_moe.NN_EARLY_STOPPING_PATIENCE = 2

    try:
        for n_phases in counts:
            chosen_formulas = all_formulas[:n_phases]
            cif_paths = [formula_to_cifs[f][0] for f in chosen_formulas]

            X_train, X_val, y_train, y_val = _build_subset_dataset(
                sec_cnn,
                cif_paths,
                synth_samples_per_formula=synth_samples_per_formula,
                random_seed=random_seed + n_phases,
            )

            torch.manual_seed(random_seed + n_phases)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(random_seed + n_phases)

            # Time the single multi-label CNN.
            t0 = time.perf_counter()
            cnn_model = sec_cnn.build_simple_nn(n_outputs=y_train.shape[1])
            sec_cnn.fit_simple_nn(cnn_model, X_train, y_train)
            cnn_elapsed = time.perf_counter() - t0

            # Time the MoE (sum of all binary experts).
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
            X_val_scaled = scaler.transform(X_val).astype(np.float32)

            t0 = time.perf_counter()
            for j in range(y_train.shape[1]):
                expert = sec_moe.build_binary_expert()
                sec_moe.fit_binary_expert(
                    expert,
                    X_train_scaled,
                    y_train[:, j],
                    X_val_scaled,
                    y_val[:, j],
                )
            moe_elapsed = time.perf_counter() - t0

            cnn_times.append(cnn_elapsed)
            moe_times.append(moe_elapsed)

            print(f"n_phases={n_phases:2d} | CNN={cnn_elapsed:7.2f}s | MoE={moe_elapsed:7.2f}s")
    finally:
        sec_cnn.NN_MAX_ITER = old_cnn_max_iter
        sec_moe.NN_MAX_ITER = old_moe_max_iter
        sec_moe.NN_EARLY_STOPPING_MIN_EPOCHS = old_moe_min_epochs
        sec_moe.NN_EARLY_STOPPING_PATIENCE = old_moe_patience

    return np.asarray(counts, dtype=int), np.asarray(cnn_times, dtype=float), np.asarray(moe_times, dtype=float)


phase_counts, cnn_times, moe_times = benchmark_cnn_vs_moe_scaling(
    phase_counts=(2, 4, 6, 8, 10),
    synth_samples_per_formula=6,
    cnn_epochs=12,
    moe_epochs=12,
)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

# Raw timing points.
ax.scatter(phase_counts, cnn_times, s=90, color="tab:red", label="CNN (single model)", zorder=3)
ax.scatter(phase_counts, moe_times, s=90, color="tab:blue", label="MoE (sum of experts)", zorder=3)

x_fit = np.linspace(phase_counts.min(), phase_counts.max(), 200)

# Exponential trend for CNN (fit in log-time space).
cnn_coef = np.polyfit(phase_counts, np.log(np.clip(cnn_times, 1e-6, None)), 1)
cnn_fit = np.exp(cnn_coef[1] + cnn_coef[0] * x_fit)
ax.plot(x_fit, cnn_fit, "--", color="tab:red", linewidth=2.2, label="CNN trend (exp fit)")

# Linear trend for MoE.
moe_coef = np.polyfit(phase_counts, moe_times, 1)
moe_fit = moe_coef[0] * x_fit + moe_coef[1]
ax.plot(x_fit, moe_fit, "--", color="tab:blue", linewidth=2.2, label="MoE trend (linear fit)")

ax.set_xlabel("Number of reference phases in training", fontsize=18, labelpad=10)
ax.set_ylabel("Training time (seconds)", fontsize=18, labelpad=10)
ax.tick_params(axis="both", labelsize=15)
ax.grid(alpha=0.25)
ax.legend(fontsize=12, framealpha=1)

out_file = Path("outputs/dl/scaling_cnn_vs_moe_time.png")
out_file.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(out_file, dpi=200)
plt.show()
print(f"Saved plot: {out_file}")


## Optional Pretrained Check


In [ ]:
PRETRAINED_PATH = "data/pretrained/moe_experts"
print("Found pretrained expert folder:", os.path.exists(PRETRAINED_PATH))

## Run the Demo


In [ ]:
# @title Run Mixture-of-Experts
create_moe_demo()

# Try on your own:
# 1) Increase expert capacity.
# create_moe_demo(conv_channels=(8, 16), hidden_layer_sizes=(32, 16))
#
# 2) Change early-stopping behavior.
# create_moe_demo(min_epochs=10, patience=6)
#
# 3) Compare shorter vs longer training budgets.
# create_moe_demo(nn_max_iter=10)
# create_moe_demo(nn_max_iter=40)


## Example Output


In [ ]:
display(Image("outputs/dl/mixture_of_experts/nn_loss_curve.png"))
display(Image("outputs/dl/mixture_of_experts/nn_test-metric_summary.png"))

## Next Steps
You have completed modules **01 through 03**.
